In [ ]:
import aprel
import numpy as np
from gymnasium_envs import make_env
from aprel.querying.value_iteration import ValueIteration
from collections import defaultdict
import osmnx as ox

python(26776) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(26777) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(26823) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


: 

In [ ]:
gym_env = make_env('GrassStreetNav-v0', place="boulder", seed=4)
env = aprel.Environment(gym_env, gym_env.feature_func)


NameError: name 'make_env' is not defined

In [ ]:
# env.reset(seed=10)

In [3]:
trajectory_set = aprel.generate_trajectories_randomly(env, random_start_state=False,
                                                        num_trajectories=2,
                                                        max_episode_length=100,
                                                        file_name='StreetNav-v0', restore=False,
                                                        headless=True, seed=0)



In [6]:
route1 = [node for node, _ in trajectory_set[0].trajectory]
route1 = [a for a, b in zip(route1, [None] + route1) if a != b]

route2 = [node for node, _ in trajectory_set[1].trajectory]
route2 = [a for a, b in zip(route2, [None] + route2) if a != b]


In [ ]:
fig, ax = ox.plot_graph_routes(gym_env.graph, [route1, route2], route_colors=['blue', 'red'],
                              route_linewidth=6, node_size=0)

In [8]:
true_user = aprel.HumanUser()

In [5]:
# Create random initial weights for a new user (we don't know their preferences yet)
params = {'weights': aprel.util_funs.get_random_normalized_vector(env.features_dim)}
user_model = aprel.SoftmaxUser(params)
# Initialize belief with empty dataset and uniform prior
belief = aprel.SamplingBasedBelief(user_model, [], params, 
                                 logprior=aprel.utils.uniform_logprior,
                                 num_samples=100)

In [6]:
belief.dataset

[]

In [7]:
query = aprel.PreferenceQuery(trajectory_set[:2])

In [8]:
query_optimizer = aprel.QueryOptimizerGen(env, env.features, horizon=env.env.horizon, seed=0)

In [9]:
queries, objective_values = query_optimizer.optimize(
    "mutual_information", belief, query,
    batch_size=1, batch_optimization_method="exhaustive_search",
    reduced_size=200, gamma=1, distance=aprel.default_query_distance,
    query_optim="querygen"
)

Optimization took 4.042487859725952 seconds.


In [19]:
queries[0].slate.trajectories[0].features

array([1.05163241e+03, 2.93418324e-01, 2.08706881e-01, 5.76725220e+02])

: 

In [14]:
route1 = queries[0].slate.trajectories[0].trajectory
route1 = [node for node, _ in route1]
route1 = [a for a, b in zip(route1, [None] + route1) if a != b]

route2 = queries[0].slate.trajectories[1].trajectory
route2 = [node for node, _ in route2]
route2 = [a for a, b in zip(route2, [None] + route2) if a != b]

In [ ]:
queries[0].reward_weights

In [ ]:
fig, ax = ox.plot_graph_routes(gym_env.graph, [route1, route2], route_colors=['blue', 'red'],
                              route_linewidth=6, node_size=0)

python3 -u examples/advanced.py --env "StreetNav-v0" --headless --simulate --num_iterations "30" \
            --acquisition "regret" --query_optim querygen --train --evaluate-test \
            --batch_size 1 --seed 1 > logs/run_1_rl_StreetNav-v0_regret.log